# ML Zoomcamp 2026 — Homework 3

Machine Learning for Classification. Target: `converted`. Implementasi mengikuti homework resmi 2026.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
TARGET = 'converted'
CATEGORICAL = ['lead_source', 'industry', 'employment_status', 'location']
NUMERICAL = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
FEATURES = CATEGORICAL + NUMERICAL

df = pd.read_csv(DATA_URL)
df.head()

## Data preparation

In [ ]:
df.isna().sum()

In [ ]:
data = df.copy()
data[CATEGORICAL] = data[CATEGORICAL].fillna('NA')
data[NUMERICAL] = data[NUMERICAL].fillna(0.0)
data.isna().sum().sum()

## Q1 — Mode of `industry`

In [ ]:
df['industry'].mode().iloc[0]

Verified answer: **`technology`** (1173 observations in the pinned release).

## Q2 — Largest correlation among the listed pairs

In [ ]:
corr = data[NUMERICAL].corr().abs()
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
pair_scores = {pair: corr.loc[pair[0], pair[1]] for pair in pairs}
pair_scores

Verified best pair: **`interaction_count` and `lead_score`** (about `0.9157` after the required missing-value fill).

## Split exactly as required

In [ ]:
df_full_train, df_test = train_test_split(data, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train.pop(TARGET)
y_val = df_val.pop(TARGET)
y_test = df_test.pop(TARGET)

len(df_train), len(df_val), len(df_test)

## Q3 — Mutual information

In [ ]:
mi_scores = {
    column: mutual_info_score(y_train, df_train[column])
    for column in CATEGORICAL
}
{k: round(v, 2) for k, v in mi_scores.items()}

Verified rounded scores are approximately: `lead_source=0.03`, `employment_status=0.02`, `industry=0.00`, `location=0.00`.

Answer: **`lead_source`**.

## Helper for Q4–Q6

In [ ]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(
        solver='liblinear', C=C, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)
    prediction = model.predict(X_val)
    return accuracy_score(y_val, prediction)

## Q4 — Validation accuracy

In [ ]:
original_accuracy = train_and_score(FEATURES, C=1.0)
original_accuracy, round(original_accuracy, 2)

Verified accuracy is about `0.645`, which rounds to **`0.65`**.

## Q5 — Feature elimination

In [ ]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
feature_differences = {}

for feature in candidates:
    reduced_features = [f for f in FEATURES if f != feature]
    reduced_accuracy = train_and_score(reduced_features, C=1.0)
    feature_differences[feature] = {
        'accuracy_without': reduced_accuracy,
        'difference': original_accuracy - reduced_accuracy,
        'absolute_difference': abs(original_accuracy - reduced_accuracy),
    }

feature_differences

Verified absolute differences are approximately `lead_source=0.003`, `number_of_courses_viewed=0.002`, `interaction_count=0.044`.

Answer: **`number_of_courses_viewed`**.

## Q6 — Tune regularization parameter `C`

In [ ]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {
    C: round(train_and_score(FEATURES, C=C), 3)
    for C in c_values
}
c_scores

Verified scores: approximately `0.598`, `0.598`, `0.613`, `0.645` respectively.

Answer: **`0.001`**.

## Final submission answers

| Question | Answer |
|---|---|
| Q1 | `technology` |
| Q2 | `interaction_count` and `lead_score` |
| Q3 | `lead_source` |
| Q4 | `0.65` |
| Q5 | `number_of_courses_viewed` |
| Q6 | `0.001` |